# 6. 卷积神经网络


## 6.1 从全连接层到卷积

> **为什么处理图像不能直接用多层感知机（MLP），而必须发明卷积？**

### 1. MLP 处理图像的“三座大山”

假设我们有一张 $1000 \times 1000$ 像素的彩色图片（3 通道）：
1.  **参数爆炸**：如果隐藏层只有 1000 个神经元，全连接层的权重矩阵大小为 $(1000 \times 1000 \times 3) \times 1000 = 3 \times 10^9$。这需要 **12GB** 显存来存储仅仅一层的权重，根本无法训练。
2.  **空间结构丢失**：MLP 要求把图片展平（Flatten）成一维。这导致像素点 $(i, j)$ 与其邻居 $(i+1, j)$ 的关系，和它与远处像素 $(i+500, j)$ 的关系在模型看来是一样的。
3.  **平移脆弱性**：如果在训练集里猫都在左上角，而测试集里的猫出现在右下角，MLP 必须重新学习一遍“右下角的猫”，它无法自动识别这种位置变化。

---

### 2. 卷积的两个物理学直觉

为了解决上述问题，科学家提出了两个核心假设：

#### 2.1. 平移不变性 (Translation Invariance)

**直觉**：不管“猫”出现在图片的哪个位置，它的特征（如尖耳朵、胡须）应该是相同的。

**数学结论**：我们的“特征检测器”在图片的任何地方都应该是**同一套权重**。

#### 2.2. 局部性 (Locality)

**直觉**：要识别一个“鼻子”，我只需要看鼻子周围的像素，不需要看图片另一角的“尾巴”。

**数学结论**：检测器每次只需要关注一个**很小的窗口**。

---

### 3. 数学演化：从全连接到卷积

#### 3.1. 全连接层公式
$$h_{i,j} = \sum_{k,l} W_{i,j,k,l} x_{k,l}$$
这里 $W$ 是一个四维张量。每一对输出坐标 $(i, j)$ 都要看遍输入坐标 $(k, l)$。

#### 3.2. 引入“平移不变性”
我们假设权重只取决于输出点 $(i, j)$ 与输入点 $(k, l)$ 的**相对距离** $(a, b)$，其中 $a = i-k, b = j-l$。
$$h_{i,j} = \sum_{a,b} V_{a,b} x_{i+a, j+b}$$
**变化**：$W$ 变成了 $V$。不管输出 $(i, j)$ 在哪，权重 $V$ 是一样的。**参数量从 $O(H^2 W^2)$ 降到了 $O(HW)$。**

#### 3.3. 引入“局部性”
我们假设当相对距离 $a$ 或 $b$ 超过一定范围（比如 3 像素）时，权重 $V_{a,b} = 0$。
$$h_{i,j} = \sum_{a=-\Delta}^{\Delta} \sum_{b=-\Delta}^{\Delta} V_{a,b} x_{i+a, j+b}$$
**变化**：求和范围缩小到一个极小的窗口。**这就是卷积！** $V$ 就是我们说的**卷积核（Kernel）**。

---

### 4. 对比 MLP 和 CNN 的参数量差异。



In [2]:
from torch import nn

def compare_parameter_counts(height: int, weight: int, channel: int) -> None:
    """对比全连接层和卷积层的参数量。
    
    Args:
        height: 图像高度。
        weight: 图像宽度。
        channel: 输入通道数。
    """
    input_dim = height * weight * channel
    hidden_dim = 1000

    # 1. 模拟全连接层
    fc = nn.Linear(input_dim, hidden_dim)
    fc_params = sum(p.numel() for p in fc.parameters())

    # 2. 模拟卷积层 (假设 3 * 3 卷积核，1000个输出通道)
    # 虽然通道数一样，但卷积核是局部共享的
    conv = nn.Conv2d(in_channels=channel, out_channels=hidden_dim, kernel_size=3)
    conv_params = sum(p.numel() for p in conv.parameters())

    print(f"图像尺寸: {height}x{weight}, 通道: {channel}")
    print(f"全连接层参数量: {fc_params:,}")
    print(f"卷积层参数量: {conv_params:,}") # 3 * 3 * 3 * 1000 + 1000
    print(f"参数压缩倍数: {fc_params / conv_params:.2f}x")

# 运行对比 (假设 128x128 的小图)
compare_parameter_counts(128, 128, 3)

图像尺寸: 128x128, 通道: 3
全连接层参数量: 49,153,000
卷积层参数量: 28,000
参数压缩倍数: 1755.46x


**实验结论**：
你会发现，对于 $128 \times 128$ 的图片，MLP 需要近 **5000 万**个参数，而卷积层只需要 **2.8 万**个。卷积层不仅效率高，而且强制模型学习“局部特征”。

---

### 5. 核心术语总结

*   **通道 (Channels)**：图像的维度。输入通常是 RGB（3通道），隐藏层可以有成百上千个通道（每个通道代表一种特征，如颜色、边缘、纹理）。
*   **感受野 (Receptive Field)**：输出中的一个像素能“看”到输入图片的范围。层数越深，感受野越大。
*   **互相关 (Cross-Correlation)**：代码里实际运行的算法，直接把卷积核 $K$ 放在 $X$ 的窗口上，对应位置相乘再求和。
*   **卷积 (Convolution)**：在进行乘法之前，先将卷积核 $K$ 水平翻转，再垂直翻转（相当于旋转 180 度），然后再执行相乘求和。但深度学习中由于核是训练出来的，翻不翻转结果一样，所以通用术语叫卷积。

---


## 6.2 图像卷积

> 从底层实现二维互相关运算，并学习如何通过数据“学习”出一个卷积核。

### 1. 二维互相关运算

在深度学习框架中，“卷积层”实际上实现的是互相关运算。

#### 1.1 运算逻辑
卷积核（Kernel）在输入张量上滑动。在每个位置，输入子张量与卷积核按元素相乘并求和，得到输出张量中的单点数值。

#### 1.2 输出形状公式
若输入形状为 $n_h \times n_w$，卷积核形状为 $k_h \times k_w$，则输出形状为：
$$(n_h - k_h + 1) \times (n_w - k_w + 1)$$

---

### 2. 手动实现卷积算子

我们将编写一个通用的互相关函数，这是所有 CNN 层的数学心脏。


In [3]:
import torch
from torch import nn, Tensor

def corr2d(X: Tensor, K: Tensor) -> Tensor:
    """计算二维互相关运算。
    
    这是卷积层最底层的数学实现。

    Args:
        X: 输入张量，形状为 (h, w)。
        K: 卷积核张量，形状为 (kh, kw)。
    
    Returns:
        Tensor: 输出特征图，形状为 (h - kh + 1, w - kw + 1)。
    """
    h, w = X.shape
    kh, kw = K.shape

    # 1. 根据公式预分配输出空间
    Y = torch.zeros((h - kh + 1, w - kw + 1), device=X.device)

    # 2. 嵌套循环实现滑动窗口
    # 注意：在生产环境（如 nn.Conv2d）中，这部分由高效的 C++/CUDA 算子完成
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            # 核心动作：切片 (Slicing) -> 逐元素相乘 -> 求和
            Y[i, j] = (X[i: i + kh, j: j+kw] * K).sum()

    return Y

# --- 简单验证 ---
X_test = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K_test = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
# 输出应为 [[19, 25], [37, 43]]
# 计算过程示例: (0*0 + 1*1 + 3*2 + 4*3) = 19
print(f"验证输出:\n{corr2d(X_test, K_test)}")

验证输出:
tensor([[19., 25.],
        [37., 43.]])


In [11]:
# 扩展

def corr2d_multi_in(X: Tensor, K: Tensor) -> Tensor:
    """处理多输入通道的互相关运算。
    
    Args:
        X: 输入张量，形状为 (c_in, h, w)。
        K: 卷积核张量，形状为 (c_in, kh, kw)。
    
    Returns:
        Tensor: 输出特征图，形状为 (h - kh + 1, w - kw + 1)。
    """

    # 逻辑：对每个输入通道分别进行互相关计算，然后将结果相加
    # zip(X, K) 会配对相应的输入通道和卷积核通道
    # 此时 x 形状为 (h, w)， k 形状为 (kh, kw)。
    # 最后将 c_in 个 2D 特征图叠加为一个 2D 特征图
    return sum(corr2d(x, k) for x, k in zip(X, K))

def corr2d_multi_in_out(X: Tensor, K: Tensor) -> Tensor:
    """处理多输入通道、多输出通道的互相关运算。
    
    Args:
        X: 输入张量，形状为 (c_in, h, w)。
        K: 卷积核张量，形状为 (c_out, c_in, kh, kw)。
    
    Returns:
        Tensor: 输出特征图，形状为 (c_out, h - kh + 1, w - kw + 1)。
    """
    # 逻辑：遍历每个输出通道对应的卷积核，计算多通道输入互相关，最后堆叠结果
    # torch.stack: 将列表中的多个张量，沿着一个新的维度拼在一起。
    # 0 表示在第一维拼接
    # 结果：将 c_out 个形状为 (h_out, w_out) 的 2D 张量，拼成一个形状为 (c_out, h_out, w_out) 的 3D 张量
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

# --- 验证样例 ---

# 1. 构造输入 X: (2个通道, 3x3 图像)
# 通道 0 全为 0, 1, 2...
# 通道 1 比通道 0 各项加 1
X_val = torch.tensor([
    [[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
    [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]
])

# 2. 构造卷积核 K: (2个输出通道, 2个输入通道, 2x2 核)
# 这里为了方便口算，我们设第一个输出通道的核全为 1，第二个输出通道的核全为 0
K_val = torch.stack([
    torch.ones((2, 2, 2)), # 输出通道 0：全 1 核
    torch.zeros((2, 2, 2)) # 输出通道 1：全 0 核
], 0)

# 3. 执行运算
Y_val = corr2d_multi_in_out(X_val, K_val)

# 4. 打印结果
print(f"输入形状: {X_val.shape}")  # (2, 3, 3)
print(f"卷积核形状: {K_val.shape}") # (2, 2, 2, 2)
print(f"输出形状: {Y_val.shape}")  # (2, 2, 2)
print(f"验证输出结果:\n{Y_val}")

输入形状: torch.Size([2, 3, 3])
卷积核形状: torch.Size([2, 2, 2, 2])
输出形状: torch.Size([2, 2, 2])
验证输出结果:
tensor([[[20., 28.],
         [44., 52.]],

        [[ 0.,  0.],
         [ 0.,  0.]]])


---

### 3. 自定义卷积层类


In [5]:
class MyConv2D(nn.Module):
    """自定义二维卷积层模块。"""

    def __init__(self, kernel_size: tuple[int, int]):
        """
        Args:
            kernel_size: 卷积核的形状 (高度, 宽度)。
        """
        super().__init__()
        # 注册可学习参数：权重 (Weight) 和 偏置 (Bias)
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, X: Tensor) -> Tensor:
        """前向传播逻辑。"""
        return corr2d(X, self.weight) + self.bias
    
# 1. 实例化自定义卷积层，传入 2x2 的卷积核尺寸
conv_layer = MyConv2D(kernel_size=(2, 2))

# 2. 准备输入数据
X_test = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])

# 3. 执行前向传播
Y_test = conv_layer(X_test)

# 4. 打印结果
print(list(conv_layer.parameters()))
print(f"输入形状: {X_test.shape}")
print(f"输出形状: {Y_test.shape}")
print(f"前向传播结果:\n{Y_test}")

[Parameter containing:
tensor([[0.8827, 0.9205],
        [0.4863, 0.4466]], requires_grad=True), Parameter containing:
tensor([0.], requires_grad=True)]
输入形状: torch.Size([3, 3])
输出形状: torch.Size([2, 2])
前向传播结果:
tensor([[ 4.1658,  6.9019],
        [12.3742, 15.1103]], grad_fn=<AddBackward0>)


In [ ]:
# 支持输入输出多通道的卷积层

class MyConv2DFull(nn.Module):
    """完整实现二维卷积层模块，支持多输入通道和多输出通道。"""
    def __init__(self, in_channels: int, out_channels: int, kernel_size: tuple[int, int]):
        """
        Args:
            in_channels: 输入通道数。
            out_channels: 输出通道数。
            kernel_size: 卷积核的形状 (高度, 宽度)。
        """
        super().__init__()
        # 1. 权重形状: (输出通道, 输入通道, 核高度, 核宽度)
        self.weight = nn.Parameter(torch.rand(out_channels, in_channels, *kernel_size))

        # 2. 偏置：每个输出通道一个偏置值
        # 形状为 (c_out, 1, 1)
        self.bias = nn.Parameter(torch.zeros(out_channels, 1, 1))
    
    def forward(self, X: Tensor) -> Tensor:
        """
        Args:
            X: 输入张量，形状为 (c_in, h, w)

        Returns:
            Tensor: 输出张量，形状为 (c_out, h_out, w_out)
        """
        return corr2d_multi_in_out(X, self.weight) + self.bias
    
# --- 验证完整卷积层 ---

# 实例化：3个输入通道（如 RGB），16个输出通道，3x3 的核
conv_layer_full = MyConv2DFull(in_channels=3, out_channels=16, kernel_size=(3, 3))

# 随机生成一个 3 通道的图像输入 (3, 32, 32)
X_input = torch.randn(3, 32, 32)

# 执行前向传播
Y_output = conv_layer_full(X_input)

print(f"输入形状: {X_input.shape}")
print(f"权重形状: {conv_layer_full.weight.shape}")
print(f"偏置形状: {conv_layer_full.bias.shape}")
print(f"输出形状: {Y_output.shape}") 
# 预期形状: (16, 32-3+1, 32-3+1) -> (16, 30, 30)

输入形状: torch.Size([3, 32, 32])
权重形状: torch.Size([16, 3, 3, 3])
偏置形状: torch.Size([16, 1, 1])
输出形状: torch.Size([16, 30, 30])


---

### 4. 手工设计边缘检测器

> 卷积核的本质是**特征提取器**。

In [7]:
# 设计一个核来寻找图像的垂直边缘。

def manual_edge_detection() -> None:
    """演示手动设计的卷积核如何提取边缘特征。"""
    # 1. 构造一个 6 x 8 的图像：中间四列为黑 (0)，两边为白 (1)
    X = torch.ones((6, 8))
    X[:, 2:6] = 0
    print("原始图像 (1=白, 0=黑):\n", X)

    # 2. 构造一个 1 x 2 的垂直边缘检测核
    # 逻辑：如果左右像素一致，结果为 0；如果不一致（边缘），结果非 0
    K = torch.tensor([[1.0, -1.0]])

    # 3. 执行运算
    Y = corr2d(X, K)
    print("\n特征图 (检测到的边缘):\n", Y)
    # 结果分析：1 代表从白变黑的边缘，-1 代表从黑变白的边缘

manual_edge_detection()

原始图像 (1=白, 0=黑):
 tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

特征图 (检测到的边缘):
 tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])


---

### 5. 从数据学习卷积核

> 深度学习可以通过反向传播让模型自己“悟”出这个核。


In [ ]:
def train_cov_kernel() -> None:
    """通过训练学习卷积核参数。"""
    # 1. 准备数据
    X = torch.ones((6, 8))
    X[:, 2:6] = 0
    K_target = torch.tensor([[1.0, -1.0]])
    Y_target = corr2d(X, K_target)

    # 2. 构造内置卷积层 (in_channels=1, out_channels=1, kernel=(1,2))
    # 注意：Conv2d 期待 4D 输入 (Batch, Channel, H, W)
    conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)

    # 将数据转换为 4D
    X_4d = X.reshape((1, 1, 6, 8))
    Y_4d = Y_target.reshape((1, 1, 6, 7))

    lr = 3e-2
    print(f">>> 目标核: {K_target}")
    print(f">>> 初始核: {conv2d.weight}")
    print("\n>>> 开始训练学习卷积核...")

    for i in range(20):
        Y_hat = conv2d(X_4d)
        # 使用均方误差 (MSE) 作为损失函数
        loss = (Y_hat - Y_4d) ** 2

        conv2d.zero_grad()
        loss.sum().backward()

        # 正常来讲这里应该要用优化器
        # 不过为了方便演示，这里采用以下写法，手动实现梯度下降。
        with torch.no_grad():
            conv2d.weight -= lr * conv2d.weight.grad

        if (i + 1) % 5 == 0:
            print(f'Epoch {i+1}, Loss: {loss.sum().item():.5f}, 当前核: {conv2d.weight.reshape(1, 2)}')

    # 3. 检查最终结果
    learned_K = conv2d.weight.reshape(1, 2)
    print(f"\n训练结束！学习到的卷积核参数:\n{learned_K}")
    print(f"误差距离: {torch.abs(learned_K - K_target).sum().item():.6f}")


train_cov_kernel()

>>> 目标核: tensor([[ 1., -1.]])
>>> 初始核: Parameter containing:
tensor([[[[ 0.0250, -0.2346]]]], requires_grad=True)

>>> 开始训练学习卷积核...
Epoch 5, Loss: 0.36634, 当前核: tensor([[ 0.9409, -0.8722]], grad_fn=<ViewBackward0>)
Epoch 10, Loss: 0.01482, 当前核: tensor([[ 0.9787, -1.0012]], grad_fn=<ViewBackward0>)
Epoch 15, Loss: 0.00131, 当前核: tensor([[ 1.0026, -0.9952]], grad_fn=<ViewBackward0>)
Epoch 20, Loss: 0.00014, 当前核: tensor([[ 0.9987, -1.0011]], grad_fn=<ViewBackward0>)

训练结束！学习到的卷积核参数:
tensor([[ 0.9987, -1.0011]], grad_fn=<ViewBackward0>)
误差距离: 0.002417


---

### 6. 细节补充


1.  **感受野 (Receptive Field)**：
    *   在卷积网络中，对于任一层的任意元素 $x$，其感受野是指在前向传播期间可能影响 $x$ 计算的所有输入。
    *   **教材要点**：我们可以通过堆叠多个小卷积核层来代替一个大卷积核层，这样不仅参数更少，而且能引入更多的非线性激活。
2.  **互相关 vs 卷积**：
    *   数学上的卷积需要将核先进行水平和垂直翻转。
    *   **工程结论**：由于卷积核本身是随机初始化并训练出来的，翻不翻转完全不影响模型的表达能力。为了代码简洁，所有深度学习框架默认实现的都是互相关。
3.  **特征图 / 特征映射 (Feature Map)**：
    *   二维卷积层的输出有时被称为特征图，因为它可以被视为输入在空间维度上提取出的特征。

---


## 6.3 填充和步幅
